# What changes when a river unit is collapsed?
## Parent–child predictors, compound split behaviour and dimensional analysis

**Question:** which changes in distance, flow allocation, branch travel-time differences and routing dispersion explain the *observed hydrograph change* between the original and collapsed states?

This is a working exploration, not a proposed final feature set. Start with physical mechanisms, inspect the paired evidence, then examine what is redundant and what survives held-out evaluation. Paths are only one mathematical tool for aggregating compound networks; individual reaches, bifurcations, matched domains and whole-network changes are also represented.

**Frozen evidence:** 552 unique comparisons across 02, 03, 05, 21 and 37, with four existing routing configurations. Aliases remain deduplicated; `island_37_151` remains excluded. The 77 audited topology groups are retained, including the 384-comparison compound in 37. Existing outputs and calibration are read only. No cell routes, recalibrates or changes production inputs.

**How to use:** Run All, then edit the controls below. The feature browser, scatter panels and case study are intended to be changed and rerun. The notebook has pre-rendered outputs and needs no widgets. Primary controls use Cunge; fixed-X controls are available for geometry-only comparison. Saved Cunge hydraulic descriptors must not be interpreted as fixed-X hydraulic predictors.

In [ ]:
from pathlib import Path
import sys, json, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID/src').exists())
sys.path.insert(0, str(ROOT/'RAPID/investigations/parent_child_predictor_exploration'))
from features import METRICS, checks, dimensional_basis
from explore import associations, scatter_grid, redundancy, fixed_panels, evaluate_fixed, COLORS
DATA_DIR = ROOT/'RAPID/outputs/parent_child_predictors_20261006T185003'
# Each Run All creates a separate analysis snapshot; the feature cache stays frozen.
RUN_DIR = DATA_DIR/('exploration_'+datetime.now().strftime('%Y%m%dT%H%M%S_%f'))
RUN_DIR.mkdir(exist_ok=False)
FIG_DIR = RUN_DIR/'figures'; FIG_DIR.mkdir()
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False, 'font.size':10})
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 30)
all_data = pd.read_parquet(DATA_DIR/'paired_candidates.parquet')
catalog = pd.read_csv(DATA_DIR/'feature_catalog.csv')
ROUTING = 'cunge'  # cunge, fixed_archived, fixed_calibrated, fixed_same_n
RESPONSE = 'relative_shift_pct'  # peak_excess_change_pct, event_duration_change_pct, shift_h, shift_over_event_T
NETWORKS = ['02','03','05','21','37']
FEATURE_FAMILY = 'splits'  # length, width, topology, splits, split_contrast, shape, hydraulics, exposure_position
FEATURE_SEARCH = 'domain_delta'  # e.g. domain_parent, network_delta, dominance, split_impact
CASE_NETWORK, CASE_STATE = '37', 'island_37_209'
RUN_FIXED_PANELS = True  # fixed exploratory panels; no response-based feature selection
MODEL = 'extra_trees'   # optional 'ridge' uses fixed alpha=10
D = all_data[all_data.method.eq(ROUTING) & all_data.example_id.isin(NETWORKS)].copy()
assert len(all_data)==2208 and not all_data.duplicated(['example_id','fingerprint','method']).any()
assert not D.state_id.eq('island_37_151').any()
print(f'Loaded {len(D)} unique pairs; {D.group.nunique()} topology groups; {len(catalog)} scoped/transformed columns from {len(METRICS)} primitive graph summaries.')
print('Analysis outputs:', RUN_DIR.relative_to(ROOT))

### 1. Start with the response, not a preferred geometric index

Peak timing, peak amplitude and duration are different outcomes. A change in average travel time can shift the hydrograph; a change in differences among branch arrival times can alter peak shape and timing; within-reach dispersion can attenuate and broaden it. The inlet waveform and downstream routing modify the effect visible at the outlet. These are hypotheses to test against the data, not established causes of every observed shift.

The primary response is
\[
y_T=100\frac{T_c-T_p}{T_p-T_i}.
\]
This is dimensionless, conditional on the known parent lag. `shift_h` retains absolute timing. `shift_over_event_T` is an additional diagnostic normalization using the inlet-event width, without replacing the established target. Peak excess and event duration retain their existing percentage definitions.

Correlated comparisons are not independent replicates. A pooled scatter is dominated by network 37; the network panels and equal-group summaries matter. Sub-reporting-interval shifts are a resolution diagnostic, not a confidence interval.

In [ ]:
coverage=D.groupby('example_id').agg(pairs=('fingerprint','size'),groups=('group','nunique'),lag_h=('parent_lag_h','first'),min_shift_pct=('relative_shift_pct','min'),max_shift_pct=('relative_shift_pct','max'),median_abs_shift_min=('shift_h',lambda s:60*s.abs().median()))
display(coverage.round(4))
fig,axes=plt.subplots(1,3,figsize=(13,3.5),layout='constrained')
for ax,y in zip(axes,['relative_shift_pct','peak_excess_change_pct','event_duration_change_pct']):
    for ex,g in D.groupby('example_id'):
        ax.scatter(g.relative_shift_pct,g[y],s=15,alpha=.55,color=COLORS[ex],label=ex)
    ax.set(xlabel='Peak timing change (%)',ylabel=y.replace('_',' '),xscale='symlog',yscale='symlog'); ax.axhline(0,color='gray',lw=.5)
axes[0].legend(title='Network',ncol=3,fontsize=8)
fig.savefig(FIG_DIR/'responses.png',bbox_inches='tight');plt.show()

### 2. Buckingham Π: choose physical quantities and scales before reducing them

Buckingham's theorem constrains a dimensionally homogeneous relationship to combinations with zero net dimensions. For a dimensional matrix of rank \(r\), \(n\) quantities admit \(n-r\) independent Π products. **It does not identify the relevant physical variables, determine a response law, or eliminate statistical collinearity.** [Buckingham's original paper](https://warwick.ac.uk/fac/sci/physics/research/cfsa/people/sandrac/lectures/buckingham1914.pdf).

The reduced convection–diffusion description uses wave celerity \(c\) and diffusivity \(D\), with dimensions \(L/T\) and \(L^2/T\). The Cunge formulation links these to reach geometry and reference hydraulics. [HEC technical reference](https://www.hec.usace.army.mil/confluence/hmsdocs/hmstrm/channel-flow/muskingum-cunge-model). Our code uses **constant reference parameters**, unlike a general variable-coefficient implementation.

For this exploration choose:
- \(L_*\): base-network allocation-weighted mean route length, fixed for all comparisons in that network;
- \(T_*\): SD of excess-inlet-flow volume in time, measured from the existing forcing and fixed before examining child responses.

Thus lengths become \(L/L_*\), times \(\tau/T_*\), diffusivities \(D T_*/L_*^2\), and changes use the **same parent/network scales**. A child/parent ratio also removes dimensions but is undefined for a zero parent (e.g. no branching); those entries stay missing. Bounded fractions and topology counts are already dimensionless.

These scale choices are hypotheses, not a unique Π basis. Event timing, flow amplitude, dimensionless waveform shape and spatial heterogeneity may still matter. Five networks/events cannot establish universal similarity.

In [ ]:
units=pd.DataFrame([
 ['length / width / depth','m',1,0],['time / lag / event width','s',0,1],
 ['celerity / water velocity','m s^-1',1,-1],['hydraulic diffusivity','m^2 s^-1',2,-1],
 ['discharge','m^3 s^-1',3,-1],['slope / split fraction / counts','1',0,0],
 ['bifurcation density','m^-1',-1,0],['Manning n (SI convention)','s m^-1/3',-1/3,1],
 ['gravity','m s^-2',1,-2]
],columns=['quantity','units','L exponent','T exponent'])
display(units)
dimensions,pi_basis=dimensional_basis();display(dimensions)
print('Dimensional rank:',np.linalg.matrix_rank(dimensions),' independent products:',len(pi_basis.columns))
display(pi_basis.T)
assert np.allclose(dimensions.to_numpy() @ pi_basis.to_numpy(),0)
display(pd.read_csv(DATA_DIR/'forcing_scales.csv',dtype={'example_id':str}).round(4))

**Important distinctions.** Manning \(n\) is dimensional under the SI Manning convention; a possible normalized form is \(n L_*^{1/3}/T_*\). If celerity is already derived from \(n,h,S\), adding all of them is not an independent physical description. The same issue applies to diffusivity and its ingredients. Froude number uses **water velocity**, \(U/\sqrt{gh}\), not automatically flood-wave celerity; the current reduced routing model does not resolve all inertial physics.

For a chosen domain-scale \(c_{eff}=\bar L/\bar\tau\) and length-weighted \(D\), inspect
\[
Pe=\frac{c_{eff}\bar L}{D},\quad Fo=\frac{D T_*}{\bar L^2},\quad C_{event}=\frac{c_{eff}T_*}{\bar L}.
\]
But \(Pe\,Fo=C_{event}\): **do not enter all three as independent log predictors**. Here these are aggregate proxies; they need not preserve the transfer behaviour of a spatially heterogeneous network. \(C_{event}\) is an event-scale transport ratio, not a numerical Courant number.

In [ ]:
pi_error=np.max(abs(D.parent_peclet_proxy*D.parent_fourier_proxy-D.parent_courant_event_proxy))
print('Max |Pe*Fo-C_event|:',pi_error)
if ROUTING=='cunge':
    fig=scatter_grid(D,['parent_peclet_proxy','domain_parent_route_time_mean_nd','domain_delta_route_time_mean_nd','exposed_delta_within_dispersion'],RESPONSE)
    fig.savefig(FIG_DIR/'dimensionless_hydraulic_diagnostics.png',bbox_inches='tight');plt.show()
else:
    print('Cunge hydraulic diagnostics skipped for the fixed-X response.')

### 3. What is measured before and after collapse?

Three scopes are deliberately separate:
1. **Matched domain:** the existing closed parent and child domains. Both states have summaries, signed changes and ratios; no cross-state link-ID matching is assumed.
2. **Whole network:** paired summaries show the integrated change and include redistribution outside a local edit. Whole-network parent values are constant within a network and can act as network identifiers, so they are not automatic candidates.
3. **Selected parent support:** the unit/island support from the audited base membership. This is a selection proxy, **not certified actual edited geometry**. There is no invented child footprint. Saved lineage omits later anchor splits in some states.

Parent context alone cannot describe the collapse. Conversely, a change without its starting state may hide whether the same change occurred in a tiny branch or a large compound. We examine parent + change, and compare that representation with change-only. Parent, child and difference are not jointly treated as independent columns.

In [ ]:
paired_metrics=['route_length_mean','dominance_mean','expected_splits','split_impact_sd','route_length_cv','route_time_mean']
if ROUTING!='cunge':paired_metrics=paired_metrics[:-1]
fig,axes=plt.subplots(2,3,figsize=(14,8),layout='constrained')
for ax,key in zip(axes.flat,paired_metrics):
    for ex,g in D.groupby('example_id'):
        ax.scatter(g['domain_parent_'+key],g['domain_child_'+key],s=14,alpha=.5,color=COLORS[ex],label=ex)
    vals=D[['domain_parent_'+key,'domain_child_'+key]].to_numpy();low=np.nanmin(vals);high=np.nanmax(vals)
    ax.plot([low,high],[low,high],color='gray',ls='--',lw=.8);ax.set(xlabel='Parent',ylabel='Child',title=key.replace('_',' '))
axes[0,0].legend(fontsize=8,ncol=3)
fig.savefig(FIG_DIR/'parent_vs_child.png',bbox_inches='tight');plt.show()

### 4. An extensive library, with a searchable definition for every variable

The primitive summaries are expanded across domain/network, parent/child/change/ratio, and dimensionless representations. The resulting columns are **alternative descriptions**, not hundreds of independent physical mechanisms. Some useful alternatives are intentionally redundant for exploration.

Families include unique physical length and area; width mean, SD, CV, harmonic mean and quantiles; bifurcation count/density, cycle rank and explicit log path count; allocation-weighted route distance moments; split dominance, effective branch count, normalized entropy, exposure and impact distributions; branch-distance contrast; and conditional reference-hydraulic travel-time/dispersion summaries.

`feature_catalog.csv` records units, scope, transformation, physical interpretation and availability. “Geometry with width allocation” means the branch probabilities are an assumption based on outgoing widths. For the saved Cunge model that is also its reference-flow allocation rule. It is not measured discharge.

In [ ]:
print('Primitive summaries:',len(METRICS),'  scoped/transformed candidates:',len(catalog))
display(catalog.groupby(['family','kind']).size().rename('columns').reset_index())
selection=catalog[catalog.family.eq(FEATURE_FAMILY)&catalog.feature.str.contains(FEATURE_SEARCH,regex=False)]
display(selection[['feature','scope','role','kind','dimensionless','definition','normalization']])
missing=D[catalog.feature].isna().mean().sort_values(ascending=False)
display(missing[missing>0].head(12).rename('missing_fraction').to_frame())
# Ratios with zero denominators and nonclosed selection supports are intentionally missing.

### 5. General split impact: local allocation and how much flow encounters it

At bifurcation \(v\), let \(p_{vj}\) be outgoing width/reference-flow fractions and \(r_v\) the probability that a particle entering the domain reaches that node.

- Dominance: \(d_v=\max_j p_{vj}\).
- Effective branches: \(b_v=1/\sum_j p_{vj}^2\).
- Normalized local entropy: \(h_v=-\sum_j p_{vj}\log p_{vj}/\log k_v\).
- A candidate general split-impact score: \(I_v=r_v(1-\sum_jp_{vj}^2)\).

We retain means and SDs. Dominance/entropy summaries use \(r_v\) as weights; split-impact mean/SD describe the collection of \(I_v\) values across physical bifurcations. \(\sum I_v\) and \(\sum r_v\) count repeated branching opportunities and may exceed one. Neither is the fraction of the inlet modified. The **domain inlet exposure** is recorded separately and is not forced to one for containing compounds.

All split summaries equal zero if no bifurcation exists; the count records that structural absence. A zero dominance mean for a single channel therefore means “no split”, not “perfectly balanced split”. Domain statistics and parent-to-child changes do not need to pair individual bifurcations across renumbered states.

In [ ]:
SPLIT_FEATURES=['domain_parent_dominance_mean','domain_delta_dominance_mean','domain_parent_split_impact_mean','domain_delta_split_impact_mean','domain_parent_split_impact_sd','domain_delta_split_impact_sd']
fig=scatter_grid(D,SPLIT_FEATURES,RESPONSE)
fig.savefig(FIG_DIR/'split_impact.png',bbox_inches='tight');plt.show()
display(D.groupby('unit_type')[SPLIT_FEATURES+[RESPONSE]].median().round(5))

### 6. Split balance alone does not tell us whether arrival times differ

Two equal-flow branches can have identical travel times, producing little between-route spreading. Another equal split can have very unequal distances or velocities. We therefore combine allocation with **branch contrast**.

For each node, recursively calculate mean remaining distance \(m_v\). At a bifurcation, compare \(L_{vj}+m_j\) across outgoing branches. Under the specified Markov split allocation, the law of total variance gives the exact identity
\[
\operatorname{Var}(L_{route})=\sum_v r_v\operatorname{Var}_{p_v}(L_{vj}+m_j).
\]
The analogous identity holds for sums of \(L_e/c_e\). The recursion handles shared downstream reaches without enumerating complete paths. This is a decomposition of the assumed route distribution, not proof that peak timing equals mean route time.

We also compute a reference-hydraulic within-route dispersion proxy \(E[\sum_e 2D_eL_e/c_e^3]\). Combined with between-route time variance it provides a low-frequency time-spread diagnostic. It does not reconstruct the hydrograph or rerun routing, and its mean/spread alone cannot determine the peak of an arbitrary forcing waveform.

In [ ]:
for prefix in ['domain_parent_','domain_child_','network_parent_','network_child_']:
    np.testing.assert_allclose(D[prefix+'branch_length_variance_sum'],D[prefix+'route_length_sd']**2,rtol=1e-8,atol=1e-4)
    np.testing.assert_allclose(D[prefix+'branch_time_variance_sum'],D[prefix+'route_time_sd']**2,rtol=1e-8,atol=1e-4)
print('Both branch-contrast decompositions verified for all parent and child states.')
CONTRAST_FEATURES=['domain_delta_route_length_mean_nd','domain_delta_route_length_cv','exposed_delta_route_time','exposed_delta_route_time_variance','exposed_delta_within_dispersion','network_delta_total_time_spread_nd']
if ROUTING!='cunge':CONTRAST_FEATURES=CONTRAST_FEATURES[:2]
fig=scatter_grid(D,CONTRAST_FEATURES,RESPONSE)
fig.savefig(FIG_DIR/'paired_mechanisms.png',bbox_inches='tight');plt.show()

### 6b. A direct physical check before fitting a statistical predictor

Let \(\bar\tau_s=E_s[\sum_e L_e/c_e]\) for state \(s\), using that state's reference split allocation. Examine
\[
\widehat{T_c-T_p}=\bar\tau_c-\bar\tau_p.
\]
This is a **mean-transit-delay change used as a peak-shift proxy**. It contains both distance changes and changes in velocities/flow allocation, handles overlapping routes by recursion, and has no fitted regression coefficient. Peak timing is not the same as mean delay; the difference is an informative residual, particularly when the waveform or route spread changes.

The calculation uses **both states' saved calibrated Cunge parameters**, not the child outlet response. This is a conditional mechanistic diagnostic, not a geometry-only or untouched out-of-network model. The scores below do not represent cross-validation of a learned predictor. Compare every network and group weighting, not just pooled R². The “without compound” row is an influence diagnostic only.

In [ ]:
from sklearn.metrics import r2_score
mechanism=all_data[all_data.method.eq('cunge')].copy()
mechanism['proxy_shift_h']=mechanism.network_delta_route_time_mean/3600
mechanism['peak_minus_mean_delay_h']=mechanism.shift_h-mechanism.proxy_shift_h
mechanism['peak_minus_mean_delay_over_T']=mechanism.peak_minus_mean_delay_h*3600/mechanism.event_T_s
mechanism_rows=[]
scopes=[('all',mechanism)]+list(mechanism.groupby('example_id'))+[('without_37_CB001_DIAGNOSTIC',mechanism[~(mechanism.example_id.eq('37')&mechanism.state_id.eq('CB001_bubble_3'))])]
for scope,g in scopes:
    for weighting in ['pair','equal_group']:
        weights=np.ones(len(g)) if weighting=='pair' else 1/g.groupby('group')['shift_h'].transform('size').to_numpy()
        mechanism_rows.append(dict(scope=scope,weighting=weighting,pairs=len(g),r2=r2_score(g.shift_h,g.proxy_shift_h,sample_weight=weights),mae_minutes=60*np.average(abs(g.shift_h-g.proxy_shift_h),weights=weights),zero_mae_minutes=60*np.average(abs(g.shift_h),weights=weights)))
mechanism_scores=pd.DataFrame(mechanism_rows);display(mechanism_scores.round(4))
mechanism_scores.to_csv(RUN_DIR/'mean_transit_delay_proxy_scores.csv',index=False)
mechanism[['example_id','fingerprint','state_id','group','shift_h','proxy_shift_h','peak_minus_mean_delay_h']].to_csv(RUN_DIR/'mean_transit_delay_proxy_cases.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
for ex,g in mechanism.groupby('example_id'):
    axes[0].scatter(g.proxy_shift_h,g.shift_h,s=17,alpha=.6,color=COLORS[ex],label=ex)
    axes[1].scatter(g.network_delta_total_time_spread_nd,g.peak_minus_mean_delay_over_T,s=17,alpha=.6,color=COLORS[ex],label=ex)
axes[0].plot([-10,3],[-10,3],ls='--',color='gray');axes[0].set(xlabel='Parameter-derived mean delay change (h)',ylabel='Observed peak shift (h)',xscale='symlog',yscale='symlog');axes[0].legend(ncol=3,fontsize=8)
axes[1].axhline(0,color='gray',lw=.7);axes[1].set(xlabel='Change in total time spread / event width',ylabel='(Peak shift − mean-delay change) / event width');axes[1].set_xscale('symlog',linthresh=.001);axes[1].set_yscale('symlog',linthresh=.001)
fig.savefig(FIG_DIR/'mean_delay_physical_proxy.png',bbox_inches='tight');plt.show()
display(mechanism[mechanism.state_id.isin(['CB001_bubble_3','island_37_209','island_37_230'])][['example_id','state_id','shift_h','proxy_shift_h','peak_minus_mean_delay_h']].round(4))

### 7. What does the current evidence associate with each response?

The table below is descriptive, not feature selection. Inspect signs and magnitudes **within each network**, alongside pair-weighted and equal-group pooled rank correlations. Equal-group correlations below reweight ordinary ranks; they are not a fully weighted empirical-CDF definition of Spearman correlation. Missing values are excluded pairwise and counts are retained.

No p-values are attached to hundreds of correlated searches. A strong association can arise from a common network scale, calibration, shared context or one influential compound. Rankings must not be used to select features and then claim the same outer folds as unbiased validation.

In [ ]:
# Bounded predeclared view; the full default-candidate library is also exported.
candidate_cols=catalog[catalog.default_candidate.astype(bool)].feature.tolist()
if ROUTING!='cunge':
    candidate_cols=catalog[catalog.feature.isin(candidate_cols)&~catalog.kind.eq('hydraulic_Cunge_reference')].feature.tolist()
A=associations(D,candidate_cols)
A.to_csv(RUN_DIR/'candidate_associations.csv',index=False)
primary=['domain_delta_route_length_mean_nd','domain_delta_width_mean_nd','domain_delta_dominance_mean','domain_delta_split_impact_mean','domain_delta_split_impact_sd','domain_delta_expected_splits','domain_delta_route_length_cv','exposed_delta_route_length','exposed_delta_route_time','exposed_delta_within_dispersion','network_delta_route_time_mean_nd','network_delta_total_time_spread_nd']
primary=[c for c in primary if c in candidate_cols]
summary=A[A.response.eq(RESPONSE)&A.feature.isin(primary)].copy();summary['view']=summary.scope+'/'+summary.weighting
association_table=summary.pivot(index='feature',columns='view',values='spearman')
display(association_table.round(3))
fig,ax=plt.subplots(figsize=(11,6),layout='constrained')
im=ax.imshow(association_table.to_numpy(),vmin=-1,vmax=1,cmap='RdBu_r',aspect='auto');ax.set_xticks(range(len(association_table.columns)),association_table.columns,rotation=40,ha='right');ax.set_yticks(range(len(association_table.index)),[x.replace('domain_','').replace('network_','net_') for x in association_table.index]);fig.colorbar(im,ax=ax,label='Rank association');ax.set_title(RESPONSE+' — consistency across networks matters')
fig.savefig(FIG_DIR/'association_consistency.png',bbox_inches='tight');plt.show()
# Removing the giant group is only an influence diagnostic, not a revised primary cohort.
giant=D.groupby('group').size().idxmax();influence=associations(D[D.group.ne(giant)],primary,[RESPONSE]);influence.to_csv(RUN_DIR/'associations_without_largest_group_DIAGNOSTIC.csv',index=False)
display(influence[influence.scope.eq('pooled')&influence.weighting.eq('equal_group')][['feature','spearman','n_valid']].round(3))

### 8. Redundancy before modelling

Several identities are structural:
- child = parent + difference; ratio is another transformation of that pair;
- route SD² = summed branch-contrast variance;
- for a connected single-outlet DAG, cycle rank = sum(outdegree−1); for exclusively binary branching it also equals bifurcation count;
- split-impact sum = number of bifurcations × split-impact mean;
- \(Pe\,Fo=C_{event}\).

These are recorded to prevent counting them as separate physical evidence. Standardization and logarithms do not make dependent features independent. Buckingham Π reduces dimensional freedom, not empirical multicollinearity.

The default panels use parent + change, not parent + child + change, and choose one representation of exact identities. Strong remaining empirical correlations are screened *inside training folds only*. The following full-data matrix is exploratory and must not be used as the fitted filter in validation. VIF/condition numbers from hundreds of columns with only 552 correlated rows are not an automatic feature-selection rule.

In [ ]:
REDUNDANCY_FEATURES=['domain_parent_route_length_mean_nd','domain_delta_route_length_mean_nd','domain_parent_width_mean_nd','domain_delta_width_mean_nd','domain_parent_width_cv','domain_delta_width_cv','domain_parent_dominance_mean','domain_delta_dominance_mean','domain_parent_split_impact_sd','domain_delta_split_impact_sd','domain_parent_expected_splits','domain_delta_expected_splits','domain_parent_route_length_cv','domain_delta_route_length_cv','domain_inlet_exposure_parent','selection_domain_ratio','upstream_distance_over_network','downstream_distance_over_network']
pairs,rank=redundancy(D,REDUNDANCY_FEATURES)
pairs.to_csv(RUN_DIR/'strong_correlations_default_panel.csv',index=False);rank.to_csv(RUN_DIR/'rank_default_panel.csv',index=False)
display(rank.round(2));display(pairs.head(25).round(3))
corr=D[REDUNDANCY_FEATURES].corr(method='spearman')
fig,ax=plt.subplots(figsize=(10,9),layout='constrained');im=ax.imshow(corr,vmin=-1,vmax=1,cmap='RdBu_r');labels=[c.replace('domain_','').replace('_nd','') for c in corr.columns];ax.set_xticks(range(len(labels)),labels,rotation=90,fontsize=8);ax.set_yticks(range(len(labels)),labels,fontsize=8);fig.colorbar(im,ax=ax,label='Spearman correlation');fig.savefig(FIG_DIR/'redundancy.png',bbox_inches='tight');plt.show()

### 9. A selected failure case: inspect the paired geometry and the actual saved hydrographs

Use the controls at the top to choose a case. The selected-unit support may be much smaller than the closed context used for resistance and path summaries. This panel avoids presenting context as the actual edit. Hydrographs are read from existing completed outputs; the case view cannot change calibration or routing.

In [ ]:
case=all_data[(all_data.example_id==CASE_NETWORK)&(all_data.state_id==CASE_STATE)&all_data.method.eq('cunge')]
assert len(case)==1,'Choose an existing canonical case';r=case.iloc[0]
case_keys=['route_length_mean','width_mean','dominance_mean','dominance_sd','split_impact_mean','split_impact_sd','expected_splits','cycle_rank','route_length_cv','route_time_mean','route_time_sd','within_route_dispersion','total_time_spread']
case_table=pd.DataFrame({'metric':case_keys,'parent':[r['domain_parent_'+k] for k in case_keys],'child':[r['domain_child_'+k] for k in case_keys],'delta':[r['domain_delta_'+k] for k in case_keys]})
display(case_table.round(5));display(case[['state_id','domain_type','selection_support_length_fraction','domain_length_fraction','domain_inlet_exposure_parent','relative_shift_pct','peak_excess_change_pct','event_duration_change_pct']].T)
source_meta=json.loads((ROOT/'RAPID/outputs/cunge_fixed_comparison_plus37_20261006/provenance.json').read_text())
src=next(s for s in source_meta['sources'] if s['example']==CASE_NETWORK)
root=Path(src['source']);parent=pd.read_parquet(root/'states'/src['base_fingerprint']/'outlet_hydrograph.parquet');child=pd.read_parquet(root/'states'/r.fingerprint/'outlet_hydrograph.parquet')
t=parent.time_seconds.to_numpy()/3600;inlet=parent.q_inflow_cms.to_numpy();qp=parent.q_outlet_cms.to_numpy();qc=child.q_outlet_cms.to_numpy()
fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
axes[0].plot(t,inlet,color='gray',alpha=.5,label='Inlet');axes[0].plot(t,qp,label='Parent');axes[0].plot(t,qc,label='Collapsed child');axes[0].set(xlabel='Time (h)',ylabel='Discharge (m³/s)',title=CASE_NETWORK+' / '+CASE_STATE);axes[0].legend()
peak_t=t[qp.argmax()];zoom=max(3,r.event_T_s/3600);axes[1].plot(t,qp,label='Parent');axes[1].plot(t,qc,label='Child');axes[1].set(xlim=(peak_t-zoom,peak_t+zoom),xlabel='Time (h)',ylabel='Discharge (m³/s)',title='Same curves around the parent peak')
fig.savefig(FIG_DIR/'case_hydrographs.png',bbox_inches='tight');plt.show()

### 10. Fixed-panel prediction checks — do paired changes help out of sample?

This is a diagnostic comparison of **predeclared panels**, not an optimized final model:
- the previous 15-feature baseline;
- paired dimensional geometry;
- paired dimensionless geometry;
- dimensionless geometry changes without parent shape;
- paired geometry plus saved Cunge hydraulic descriptors.

The last panel is conditional on calibrated reference hydraulics and is not an ungauged geometric model. The dimensional and dimensionless geometry panels retain the same split/context features; their dimensional length/width variables use different network-scale representations.

Five grouped folds and five whole-network folds answer different questions. The giant compound remains indivisible. Training stays pair-weighted; evaluation reports pair-, equal-group- and equal-network weighting. Within each training fold, median imputation, constant removal and fixed-order |Spearman|≥0.97 pruning are fitted without test rows or target rankings. The legacy baseline is unchanged. Extra Trees uses its previous fixed configuration; optional Ridge uses fixed α=10 and training-only scaling.

There is **no selection or tuning here**. If these results guide a feature choice, a subsequent nested grouped procedure must evaluate that selection, and reuse of this cohort remains exploratory. No SHAP or importance plot can convert this comparison into physical causation.

In [ ]:
if RUN_FIXED_PANELS:
    assert ROUTING=='cunge' and len(NETWORKS)==5,'Full fixed-panel comparison uses the frozen five-network Cunge cohort.'
    legacy_meta=json.loads((ROOT/'RAPID/outputs/cunge_fixed_extra_trees_plus37_20261006/provenance.json').read_text())
    legacy_cols=legacy_meta['geometry_features'];old=pd.read_parquet(ROOT/'RAPID/outputs/cunge_fixed_extra_trees_plus37_20261006/matched_features_responses.parquet')
    old=old[old.method.eq('cunge')][['example_id','fingerprint']+legacy_cols]
    extra=[c for c in legacy_cols if c not in D.columns]
    modelling=D.merge(old[['example_id','fingerprint']+extra],on=['example_id','fingerprint'],validate='one_to_one')
    panels=fixed_panels(legacy_cols)
    P,S,F=evaluate_fixed(modelling,panels,target=RESPONSE,model=MODEL)
    P.to_parquet(RUN_DIR/'fixed_panel_predictions.parquet',index=False);S.to_csv(RUN_DIR/'fixed_panel_scores.csv',index=False);F.to_csv(RUN_DIR/'training_fold_filters.csv',index=False)
    units='hours' if RESPONSE=='relative_shift_pct' else 'target'
    view=S[S.scope.eq('pooled')&S.units.eq(units)].copy()
    if units=='hours':view['mae_minutes']=60*view.mae;view['zero_mae_minutes']=60*view.zero_mae
    display(view.round(4))
    if MODEL=='extra_trees' and RESPONSE=='relative_shift_pct':
        oldpred=pd.read_parquet(ROOT/'RAPID/outputs/cunge_fixed_extra_trees_plus37_20261006/held_out_predictions.parquet')
        oldpred=oldpred[oldpred.routing.eq('cunge')&oldpred.response.eq(RESPONSE)&oldpred.features.eq('geometry_15')]
        replay=P[P.validation.eq('network')&P.panel.eq('legacy15')].merge(oldpred,on=['example_id','fingerprint'],suffixes=('_new','_old'),validate='one_to_one')
        np.testing.assert_allclose(replay.predicted_new,replay.predicted_old,rtol=1e-11,atol=1e-12)
        print('Legacy network predictions reproduced.')
    fig,axes=plt.subplots(1,2,figsize=(13,4),layout='constrained')
    for ax,weight in zip(axes,['pair','equal_group']):
        plot=view[view.weighting.eq(weight)].pivot(index='panel',columns='validation',values='mae')
        if units=='hours':plot*=60
        plot.plot.barh(ax=ax);ax.set(xlabel='MAE (minutes)' if units=='hours' else 'MAE (target units)',title=weight.replace('_',' '));ax.legend(fontsize=8)
    fig.savefig(FIG_DIR/'fixed_panel_scores.png',bbox_inches='tight');plt.show()
    display(S[S.units.eq(units)&S.scope.ne('pooled')&S.weighting.eq('pair')].pivot_table(index=['validation','panel'],columns='scope',values='mae').round(4))
else:
    print('Fixed-panel fitting disabled. All descriptive exploration remains available.')

### 11. Geometry-only comparison across the existing routing controls

The same geometric change need not have the same routed effect under fixed-X and Cunge. Compare the directions of association across methods before describing a relationship as general. Archived fixed-X also differs in forcing/history/grid, so it is not a clean X-only experiment.

In [ ]:
control_features=['domain_delta_route_length_mean_nd','domain_delta_width_mean_nd','domain_delta_dominance_mean','domain_delta_split_impact_sd','domain_delta_expected_splits','domain_delta_route_length_cv']
control_rows=[]
for method,g in all_data.groupby('method'):
    aa=associations(g,control_features,[RESPONSE]);aa['routing']=method;control_rows.append(aa)
control=pd.concat(control_rows,ignore_index=True);control.to_csv(RUN_DIR/'routing_control_associations.csv',index=False)
display(control[control.scope.eq('pooled')&control.weighting.eq('equal_group')].pivot(index='feature',columns='routing',values='spearman').round(3))

### 12. Checks and next decisions

Tests cover an analytic two-branch example, shared downstream reaches, invariance to degree-two subdivision, changes of geometric units, no-split conventions, and a full-rank dimensionless basis. The variance identities were also checked for every extracted graph. These establish the implemented definitions; they do not establish that a candidate controls hydrograph response.

Use this notebook to decide:
1. Does a variable describe a physical *change*, its starting state, its exposure or its surrounding context?
2. Does its association persist within networks and across the two validation questions?
3. Does the dimensionless version transfer better, or merely encode a network/event denominator?
4. Is it redundant mathematically, empirically, or through shared calibration?
5. Does a failure case show mean travel-time change, branch-time spread, within-reach dispersion, or a peak-shape effect that these summaries miss?

The next feature-selection stage should be nested inside the training groups, with a small predeclared family menu and equal-group loss. Do not force seven predictors yet. Actual-edit geometry still requires resolving post-lineage anchor splits; this notebook does not conceal that gap with a containing-compound descriptor.

In [ ]:
display(pd.Series(checks(),name='passed').to_frame())
from test_features import run as run_analytic_hydraulic_tests
run_analytic_hydraulic_tests()
provenance=json.loads((DATA_DIR/'provenance.json').read_text())
changed=[p for p,h in provenance['source_hashes'].items() if hashlib.sha256(Path(p).read_bytes()).hexdigest()!=h]
assert not changed,changed
helper_hashes={str(ROOT/'RAPID/investigations/parent_child_predictor_exploration'/name):hashlib.sha256((ROOT/'RAPID/investigations/parent_child_predictor_exploration'/name).read_bytes()).hexdigest() for name in ['features.py','explore.py','make_notebook.py','test_features.py','execute_notebook.py']}
(RUN_DIR/'notebook_run.json').write_text(json.dumps({'data':str(DATA_DIR),'routing':ROUTING,'response':RESPONSE,'networks':NETWORKS,'model':MODEL,'fixed_panels':RUN_FIXED_PANELS,'source_hashes_unchanged':True,'helper_hashes':helper_hashes,'scope':'Exploratory fixed panels and descriptive associations; no selection-adjusted performance claim, rerouting or recalibration.'},indent=2))
print(f'All {len(provenance["source_hashes"])} source hashes unchanged. Outputs: {RUN_DIR.relative_to(ROOT)}')